In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "430_build_gold_vw_dim_item_specification.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.silver.dim_item_specification"
TARGET_VIEW = f"{catalog}.gold.vw_dim_item_specification"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}',
            current_timestamp()
        )
    """)

# ============================================================
# Build gold.vw_dim_item_specification
# ============================================================
try:
    spark.sql(f"DROP VIEW IF EXISTS {TARGET_VIEW}")

    spark.sql(f"""
    CREATE VIEW {TARGET_VIEW} AS
    SELECT
          item_specification_key              AS `Item Specification Key`
        , specification_code                  AS `Specification Code`
        , specification_description           AS `Specification Description`
        , effective_specification_description AS `Effective Specification Description`
        , measurement_unit                    AS `Measurement Unit`
        , default_bid_code                    AS `Default Bid Code`
        , default_bid_item_description        AS `Default Bid Item Description`
        , default_spec_book_year              AS `Default Spec Book Year`
        , has_multiple_bid_codes              AS `Has Multiple Bid Codes`
        , has_multiple_bid_item_descriptions  AS `Has Multiple Bid Item Descriptions`
        , has_multiple_spec_book_years        AS `Has Multiple Spec Book Years`
        , vendor_project_count                AS `Vendor Project Count`
        , estimate_project_count              AS `Estimate Project Count`
    FROM {SOURCE_TABLE}
    """)

    row_count = spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {TARGET_VIEW}
    """).collect()[0]["row_count"]

    log_run("SUCCESS", row_count, "Built gold.vw_dim_item_specification successfully.")

    print("=" * 70)
    print("Built gold.vw_dim_item_specification")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise